# Phase 1 at full scale — Bengali 500M tokens + Assamese max-available

Runs the **same pipeline code** as the local `sampled` profile, with the caps removed.
Nothing here is Colab-specific except mounting Drive and installing dependencies.

**Why Colab rather than a laptop:** the Bengali ingest is ~13.5 hours of CPU
(measured at 38k chars/sec locally) and streams ~30 GB of Parquet. A typical
student laptop with 7 GB RAM will OOM on the deduplication index long before it finishes.

**The run is checkpointed.** Colab disconnects; that is expected, not a failure.
Checkpoints live on Drive, so re-running the ingest cell after a disconnect
continues where it stopped. A resumed run has been verified to produce a
byte-identical corpus to an uninterrupted one, including after `kill -9`.

**Runtime → Change runtime type → CPU.** This stage is I/O and CPU bound;
a GPU does nothing for it and burns your GPU quota, which you need for Phase 2.


## 1 · Mount Drive

Everything durable — corpora, checkpoints, tokenizers — is written under Drive.
The Colab VM's own disk is erased when the session ends.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib

# Root for all durable artifacts. Change if you keep the project elsewhere.
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/lma_project')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

!df -h /content/drive/MyDrive | tail -1


## 2 · Clone the repository

GitHub Classroom repos are **private**, so an unauthenticated clone fails in Colab.
Authenticate with a personal access token:

1. github.com → **Settings → Developer settings → Personal access tokens → Tokens (classic)**
2. **Generate new token**, tick the **`repo`** scope, copy it
3. In Colab: **🔑 sidebar → Add new secret**, name it `GH_TOKEN`, paste the value,
   and enable *Notebook access*

Keeping the token in Colab secrets rather than in a cell means it is not exposed
if you share or submit this notebook.


In [ ]:
REPO_URL = 'https://github.com/CL3-410/individual-project-Karan-thayat.git'
BRANCH   = 'phase-1'

import pathlib, subprocess
REPO = pathlib.Path('/content/lma')

try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
    print('No GH_TOKEN secret found. A private repo will fail to clone;')
    print('either add the secret, or use the upload fallback below.')

# Inject the token into the URL only at call time, so it never appears in the
# notebook source or in the printed output.
auth_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL

if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=False)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=False)
    subprocess.run(['git', '-C', str(REPO), 'pull', 'origin', BRANCH], check=False)
    print('updated existing clone')
else:
    result = subprocess.run(
        ['git', 'clone', '--branch', BRANCH, auth_url, str(REPO)],
        capture_output=True, text=True,
    )
    # Scrub the token from any error message before printing it.
    output = (result.stdout + result.stderr).replace(token, '***') if token else result.stdout + result.stderr
    print(output.strip() or 'cloned')

print('code present:', (REPO / 'lma' / 'data' / 'sources.py').exists())


**Upload fallback.** Zip the project locally (excluding `.venv`, `data/`, and
`.git`) and upload it here:

```bash
# run this on your laptop, in the project directory
zip -r lma.zip . -x '.venv/*' '*/data/*' '.git/*' '*.pyc' '__pycache__/*'
```


In [ ]:
# Only needed if the clone above did not work.
import pathlib, shutil
REPO = pathlib.Path('/content/lma')

if not (REPO / 'lma' / 'data' / 'sources.py').exists():
    from google.colab import files
    up = files.upload()          # select lma.zip
    name = next(iter(up))
    REPO.mkdir(parents=True, exist_ok=True)
    !unzip -q -o {name} -d {REPO}

    # A zip made from the parent directory nests everything one level deeper.
    # Detect that and lift the contents up, so later cells find the expected paths.
    if not (REPO / 'lma').exists():
        nested = [d for d in REPO.iterdir() if d.is_dir() and (d / 'lma').exists()]
        if nested:
            for item in nested[0].iterdir():
                shutil.move(str(item), str(REPO / item.name))
            print('lifted contents out of', nested[0].name)

assert (REPO / 'lma' / 'data' / 'sources.py').exists(), \
    'Project code not found under /content/lma - check the zip layout'
print('Code ready at', REPO)


## 3 · Install dependencies

Phase 1 needs no PyTorch — `sentencepiece` trains the tokenizers and `datasets`
is only a download client. Neither is a pretrained model.


In [ ]:
%pip install -q sentencepiece datasets pyyaml beautifulsoup4 lxml regex 2>&1 | tail -2

import sentencepiece, datasets
print('sentencepiece', sentencepiece.__version__)
print('datasets     ', datasets.__version__)


## 4 · Hugging Face token (recommended)

Unauthenticated hub requests are rate-limited, which matters when pulling ~30 GB
across 77 shards. A free read token removes that ceiling.

1. https://huggingface.co/settings/tokens → **New token** → type **Read**
2. In Colab: **🔑 (left sidebar) → Add new secret**, name it `HF_TOKEN`, paste the value,
   and enable *Notebook access*.

Using the secrets panel keeps the token out of the notebook, so it is not exposed
if you share this file.

The Sangraha and Wikipedia datasets used here are public — the token only raises
your rate limit. It becomes mandatory only if you enable the gated OSCAR source.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF token loaded from Colab secrets')
except Exception as exc:
    print('No HF_TOKEN secret found — continuing unauthenticated (slower, rate-limited).')
    print('Reason:', type(exc).__name__)

# Cache datasets on Drive so a reconnect does not re-download shards already fetched.
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'hf_cache')
print('HF cache:', os.environ['HF_HOME'])


## 5 · Point the pipeline at Drive

Two edits to the checked-in configs, both mechanical:

1. **Widen the Sangraha shard pattern** from one shard to all 77. This is the single
   change that takes the corpus from ~3.5M to ~500M tokens.
2. **Raise the download caps.** The local run capped downloaded sources so manual
   collection could clear 20%; at full scale the manual share has to be re-established
   from actual scrape/OCR volume instead (see the note in §8).


In [ ]:
import yaml, pathlib

def scale_up(manifest_path, hf_edits):
    """Widen shard patterns and drop document caps for a full-scale run.

    Rewriting via yaml.safe_dump discards the manifest's explanatory comments,
    so the annotated original is kept alongside as `.sampled.yaml` — it is the
    documentation of why each source is configured the way it is, and it is what
    you restore to go back to the bounded local run.
    """
    manifest_path = pathlib.Path(manifest_path)
    backup = manifest_path.with_suffix('.sampled.yaml')
    if not backup.exists():
        backup.write_text(manifest_path.read_text())
        print('kept annotated original at', backup.name)

    cfg = yaml.safe_load(manifest_path.read_text())
    for src in cfg['sources']:
        edit = hf_edits.get(src['source_id'])
        if edit is None:
            continue
        src.pop('limit', None)          # no per-source document cap at full scale
        if edit:                         # a shard glob, where the source needs one
            src['data_files'] = edit
    manifest_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
    return cfg

cfg = scale_up(REPO / 'bengali/configs/sources.yaml', {
    'sangraha_bn':  'verified/ben/*.parquet',   # all 77 shards, ~30 GB
    'wikipedia_bn': None,                        # all of Bengali Wikipedia
})

for s in cfg['sources']:
    if s.get('enabled', True):
        print(f"  {s['source_id']:32s} {s.get('data_files') or s.get('config') or s.get('path','')}")


## 6 · Run the Bengali ingest

**This is the long cell.** Expect roughly 6–10 hours on Colab; it will very likely
outlive at least one disconnect.

### When Colab disconnects

Re-run **this same cell**. `--resume` picks up the checkpoint from Drive: it
reconciles the output files against the checkpoint, restores the deduplication
index, and skips the documents each source already yielded. Repeat until it prints
`Corpus build complete`.

A checkpoint is written every 25,000 kept documents. `--checkpoint-dir` is on Drive
specifically so it survives the VM being recycled.


In [ ]:
import subprocess, sys

BN_OUT  = DRIVE_ROOT / 'bengali'
BN_CKPT = BN_OUT / 'checkpoints'
BN_OUT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(REPO / 'scripts/build_corpus.py'),
    '--config',  str(REPO / 'bengali/configs/pipeline.yaml'),
    '--sources', str(REPO / 'bengali/configs/sources.yaml'),
    '--profile', 'full',
    '--target-tokens', '500000000',
    '--checkpoint-dir', str(BN_CKPT),
    '--checkpoint-every', '25000',
    '--resume',            # safe on a first run: no checkpoint just means start fresh
    '--report-every', '20000',
]
print(' '.join(cmd), '\n')

# Stream output live so progress is visible rather than buffered to the end.
proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
print('\nexit code:', proc.wait())


### Checking progress without interrupting the run

Run this in a **separate cell** while the ingest is going, or after a disconnect to
see how far it got.


In [ ]:
import json

state_file = BN_CKPT / 'bn_ingest_state.json'
if state_file.exists():
    st = json.load(open(state_file))
    target_chars = 500_000_000 * 4.0
    print(f"segment      : {st['segments']}")
    print(f"docs seen    : {st['seen']:,}")
    print(f"docs kept    : {st['kept']:,}")
    print(f"train chars  : {st['train_chars']:,}  ({100 * st['train_chars'] / target_chars:.1f}% of budget)")
    print(f"elapsed      : {st['elapsed_seconds'] / 3600:.1f} h")
    print(f"sources done : {st['completed_sources']}")
else:
    print('No checkpoint yet — either the run has not saved one, or it already finished and cleared it.')


## 7 · Run the Assamese ingest

Same code, same 500M target — but Assamese public text is roughly two orders of
magnitude scarcer, so this will **exhaust its sources and stop short**. That is the
expected outcome, and the assignment permits it provided the shortfall is reported.
`scripts/tokenize_corpus.py` prints the exact achieved count and gap in §9.

Much faster than Bengali — expect 1–2 hours.


In [ ]:
cfg = scale_up(REPO / 'assamese/configs/sources.yaml', {
    'sangraha_as':  'verified/asm/*.parquet',
    'wikipedia_as': None,
})
for s in cfg['sources']:
    if s.get('enabled', True):
        print(f"  {s['source_id']:32s} {s.get('data_files') or s.get('config') or s.get('path','')}")

AS_OUT  = DRIVE_ROOT / 'assamese'
AS_CKPT = AS_OUT / 'checkpoints'
AS_OUT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(REPO / 'scripts/build_corpus.py'),
    '--config',  str(REPO / 'assamese/configs/pipeline.yaml'),
    '--sources', str(REPO / 'assamese/configs/sources.yaml'),
    '--profile', 'full',
    '--target-tokens', '500000000',
    '--checkpoint-dir', str(AS_CKPT),
    '--checkpoint-every', '25000',
    '--resume',
]
proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
print('\nexit code:', proc.wait())


## 8 · Copy corpora to Drive, and re-check the manual share

⚠️ **The 20% manual-collection requirement needs re-establishing at full scale.**

The local run cleared it (25.5% Bengali / 36.2% Assamese) because downloaded sources
were capped. Removing those caps in §5 multiplies the downloaded side by ~100× while
the scraped side stays fixed — so the manual share will collapse well below 20%
unless you scrape and OCR far more material.

The build prints the achieved share and warns when it is under 20%. Three ways to fix it:

- **Raise `max_pages`** on the Wikisource scrape sources (they cache, so re-runs are incremental)
- **Enable the `manual_scrape_custom_*` templates** with sites you are entitled to crawl
- **OCR scanned books** into `<lang>/data/raw/pdf/` and run `scripts/run_ocr.py`

Alternatively, keep a cap on the downloaded sources so the ratio holds — a smaller
corpus that satisfies the requirement beats a larger one that does not.


In [ ]:
import shutil

for lang, out in [('bn', BN_OUT), ('as', AS_OUT)]:
    src_dir = REPO / ('bengali' if lang == 'bn' else 'assamese') / 'data/processed'
    for split in ['train', 'validation', 'test']:
        f = src_dir / f'{lang}_{split}.jsonl'
        if f.exists():
            dest = out / f.name
            shutil.copy2(f, dest)
            print(f'{dest}  {dest.stat().st_size / 1e9:.2f} GB')

for lang, name in [('bn', 'bengali'), ('as', 'assamese')]:
    stats = REPO / name / 'reports/corpus_stats.json'
    if stats.exists():
        d = json.load(open(stats))
        share = d['collection_split']['manual_word_fraction']
        flag = 'OK' if share >= 0.20 else 'BELOW REQUIREMENT'
        print(f"{name:9s} manual share {100 * share:5.1f}%  [{flag}]")


## 9 · Tokenizers, token counts, and the report

Tokenizer training is fast even on a 500M-token corpus — SentencePiece samples
`input_sentence_size` lines rather than reading everything.


In [ ]:
import subprocess, sys

def run(script, *extra):
    """Run a pipeline script in the repo, streaming its output live."""
    cmd = [sys.executable, str(REPO / 'scripts' / script), *extra]
    proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    return proc.wait()

for name in ['bengali', 'assamese']:
    cfg = f'{name}/configs/pipeline.yaml'
    print(f'\n=== {name}: tokenizer sweep + training ===')
    run('train_tokenizer.py', '--config', cfg, '--profile', 'full',
        '--sweep', '--use-sweep-best')
    print(f'\n=== {name}: exact token count ===')
    run('tokenize_corpus.py', '--config', cfg)


In [ ]:
# Verification and report. validate_independence.py exits non-zero on any
# violation, so its return code is worth checking rather than just reading.
print('=== component self-tests ===')
run('test_pipeline.py')

print('\n=== independence validation ===')
rc = run('validate_independence.py')
print('independence exit code:', rc, '(0 = all checks passed)')

print('\n=== report ===')
run('make_report.py')

print('\n=== Phase 1 checklist ===')
run('check_phase1.py')


## 10 · Save everything back to Drive

Tokenizers, statistics, and the report are small — commit these to git. The corpora
are large: keep them on Drive and put the shareable links in your README, as the
assignment requires.


In [ ]:
import shutil

for name in ['bengali', 'assamese']:
    for sub in ['tokenizer', 'reports']:
        src = REPO / name / sub
        if src.exists():
            dst = DRIVE_ROOT / name / sub
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print('saved', dst)

if (REPO / 'report').exists():
    shutil.copytree(REPO / 'report', DRIVE_ROOT / 'report', dirs_exist_ok=True)
    print('saved', DRIVE_ROOT / 'report')

print('\nNow: right-click each folder in Drive → Share → "Anyone with the link",')
print('and paste the links into the README table so graders can open them without asking.')
